In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import iberoSignalPro.preprocesa as ib
from sklearn.preprocessing import StandardScaler
import os

In [2]:
ch_names = np.array(['FC3', 'FCz', 'FC4', 'CP3', 'C3', 'C1', 'Cz', 'C2', 'C4', 'CP4', 'P3', 'Pz', 'P4', 'O1', 'Oz', 'O2'], dtype=object)


In [3]:
def obtener_win(sig, binary_sig, siPlot=True):
        # Ensure binary_sig is binary
        binary_sig = np.array(binary_sig).flatten()
        binary_sig = (binary_sig >= 0.5).astype(int)  # Convert to binary (0 or 1)

        diff = np.diff(binary_sig)
        idx_actividad = np.where(diff == 1)[0]
        idx_rep = np.where(diff == -1)[0]

        print(binary_sig.shape)

        if binary_sig[0] == 1:
            idx_actividad = np.insert(idx_actividad, 0, 0)
        #if binary_sig[-1] == 1:
        #    idx_actividad = np.append(idx_actividad, len(binary_sig) -1)
        
        if binary_sig[0] == 0:
            idx_rep = np.insert(idx_rep, 0, 0)
        #if binary_sig[-1] == 0:
        #    idx_rep = np.append(idx_rep, len(binary_sig) - 1)
        
        #print(idx_actividad)
        #print(idx_rep)
        
        # Ensure idx_actividad and idx_rep have the same length by adding samples
        while len(idx_actividad) < len(idx_rep):
            idx_actividad = np.append(idx_actividad, idx_actividad[-1])
        while len(idx_rep) < len(idx_actividad):
            idx_rep = np.append(idx_rep, idx_rep[-1])
        
        if idx_rep[0] < idx_actividad[0]:
            ventanas_reposo = np.stack((idx_rep, idx_actividad)).T
            ventanas_actividad = np.stack((idx_actividad[:-1], idx_rep[1:])).T
        else:
            ventanas_reposo = np.stack((idx_rep[:-1], idx_actividad[1:])).T
            ventanas_actividad = np.stack((idx_actividad[:], idx_rep[:])).T

        #print(ventanas_reposo.shape)
        #print(ventanas_actividad.shape)

        if siPlot:
            plt.figure(figsize=(20, 5))
            plt.subplot(1, 2, 1) 
            for ventana in ventanas_actividad:
                plt.plot(sig[ventana[0]: ventana[1]])
            plt.title('Ventanas de actividad')

            plt.subplot(1, 2, 2)  
            for ventana in ventanas_reposo:
                plt.plot(sig[ventana[0]: ventana[1]])
            plt.title('Ventanas de reposo')

            plt.show()

        return ventanas_actividad, ventanas_reposo



In [4]:

def obtener_promedios_ventana(sig, ventanas_actividad, output='mean', log = False):
    promedios = []
    signal = sig[ventanas_actividad[0][0]: ventanas_actividad[0][1]]

    for ventana in ventanas_actividad[1:]:
        
        
        if ventana[0] < ventana[1]:
            signal = np.concatenate((signal, sig[ventana[0]: ventana[1]]), axis=0)
         
    promedios = np.array(signal)
    if output == 'mean':
        promedios = np.nanmean(promedios, axis=0)
    elif output == 'std':
        promedios = np.nanstd(promedios, axis=0)
    elif output == 'median':
        promedios = np.median(promedios, axis=0)
    
    if log:
        promedios = 20 * np.log10(promedios)
    return promedios, signal

In [5]:

def get_means(file_path):
    df = pd.read_csv(file_path)
    
    ventanas_actividad, ventanas_reposo = obtener_win(df["Torque"], df["Binaria"], siPlot=False)
    mu = df.iloc[:, 8::4].values
    beta = df.iloc[:, 9::4].values
    gamma = df.iloc[:, 10::4].values
    
    print("mu", df.columns[8::4])
    print("beta", df.columns[9::4])
    print("gamma", df.columns[10::4])

    #scaler = StandardScaler()
    #mu = scaler.fit_transform((mu))
    #beta = scaler.fit_transform((beta)) 
    #gamma = scaler.fit_transform((gamma)) 
    
    output = "mean"
    log = False

    mu_act, _ = obtener_promedios_ventana(mu, ventanas_actividad, output=output, log=log)
    mu_rep, _ = obtener_promedios_ventana(mu, ventanas_reposo, output=output, log=log)

    beta_act,_ = obtener_promedios_ventana(beta, ventanas_actividad, output=output, log=log)
    beta_rep,_ = obtener_promedios_ventana(beta, ventanas_reposo, output=output, log=log)

    gamma_act,_ = obtener_promedios_ventana(gamma, ventanas_actividad,  output=output, log=log)
    gamma_rep,_ = obtener_promedios_ventana(gamma, ventanas_reposo, output=output, log=log)

    return mu_act, mu_rep, beta_act, beta_rep, gamma_act, gamma_rep

In [6]:
def read_csvs(folder_path, carga):
    
    files = os.listdir(folder_path)

    csv_files = [file for file in files if file.endswith('.csv')]
    if not csv_files:
        print("No se encontraron archivos csv")
        return None, None, None, None, None, None
    
    for csv_file in csv_files:
        file_path = os.path.join(folder_path, csv_file)
        if csv_file == "10deTorquePre.csv":
            if carga == 10:
                print("\t|-10")
                mu_act, mu_rep, beta_act, beta_rep, gamma_act, gamma_rep = get_means(file_path=file_path)
            
        elif csv_file == "5deTorquePre.csv":
            if carga == 5:
                print("\t|-5")
                mu_act, mu_rep, beta_act, beta_rep, gamma_act, gamma_rep = get_means(file_path=file_path)
            
        elif csv_file == "pasivoPre.csv":
            if carga == 0:
                print("\t|-pasivo")
                mu_act, mu_rep, beta_act, beta_rep, gamma_act, gamma_rep = get_means(file_path=file_path)
   
    return mu_act, mu_rep, beta_act, beta_rep, gamma_act, gamma_rep

In [7]:
def obtenerpots(carga):
    mu_act = []
    mu_rep = []
    beta_act = []
    beta_rep = []
    gamma_act = []
    gamma_rep = []

    root_folder_path = rf"E:\Pruebas%20BCI\Completos10Hz"

    sub_sub_folders = []

    items_in_root = os.listdir(root_folder_path)

    sub_folders = [item for item in items_in_root if os.path.isdir(os.path.join(root_folder_path, item))]
    for sub_folder in sub_folders:
        sub_folder_path = os.path.join(root_folder_path, sub_folder)
        items_in_sub_folder = os.listdir(sub_folder_path)
        for item in items_in_sub_folder:
            item_path = os.path.join(sub_folder_path, item)
            if os.path.isdir(item_path):
                sub_sub_folders.append(item_path)
                try:
                    print(item_path)
                    tmu_act, tmu_rep, tbeta_act, tbeta_rep, tgamma_act, tgamma_rep = read_csvs(item_path, carga)
                    if tmu_act is None or tmu_rep is None or tbeta_act is None or tbeta_rep is None or tgamma_act is None or tgamma_rep is None:
                        continue
                    mu_act.append(tmu_act)
                    mu_rep.append(tmu_rep)
                    beta_act.append(tbeta_act)
                    beta_rep.append(tbeta_rep)
                    gamma_act.append(tgamma_act)
                    gamma_rep.append(tgamma_rep)

                    
                except:
                    print(f"******************************** Error en {item_path}")
                    continue
    return np.array(mu_act), np.array(mu_rep), np.array(beta_act), np.array(beta_rep), np.array(gamma_act), np.array(gamma_rep)

In [8]:

mu_act5, mu_rep5, beta_act5, beta_rep5, gamma_act5, gamma_rep5 = obtenerpots(5)
mu_act10, mu_rep10, beta_act10, beta_rep10, gamma_act10, gamma_rep10 = obtenerpots(10)
mu_act0, mu_rep0, beta_act0, beta_rep0, gamma_act0, gamma_rep0 = obtenerpots(0)

E:\Pruebas%20BCI\Completos10Hz\AlejandoPayan\S1
	|-5
(2664,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Index(['beta_FC3', 'beta_FCz', 'beta_FC4', 'beta_CP3', 'beta_C3', 'beta_C1',
       'beta_Cz', 'beta_C2', 'beta_C4', 'beta_CP4', 'beta_P3', 'beta_Pz',
       'beta_P4', 'beta_O1', 'beta_Oz', 'beta_O2'],
      dtype='object')
gamma Index(['gamma_FC3', 'gamma_FCz', 'gamma_FC4', 'gamma_CP3', 'gamma_C3',
       'gamma_C1', 'gamma_Cz', 'gamma_C2', 'gamma_C4', 'gamma_CP4', 'gamma_P3',
       'gamma_Pz', 'gamma_P4', 'gamma_O1', 'gamma_Oz', 'gamma_O2'],
      dtype='object')
E:\Pruebas%20BCI\Completos10Hz\AlejandoPayan\S2
	|-5
(3002,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Inde

C:\Users\fercy\AppData\Local\Temp\ipykernel_6156\3322384139.py:13: RuntimeWarning: Mean of empty slice
  promedios = np.nanmean(promedios, axis=0)


(3420,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Index(['beta_FC3', 'beta_FCz', 'beta_FC4', 'beta_CP3', 'beta_C3', 'beta_C1',
       'beta_Cz', 'beta_C2', 'beta_C4', 'beta_CP4', 'beta_P3', 'beta_Pz',
       'beta_P4', 'beta_O1', 'beta_Oz', 'beta_O2'],
      dtype='object')
gamma Index(['gamma_FC3', 'gamma_FCz', 'gamma_FC4', 'gamma_CP3', 'gamma_C3',
       'gamma_C1', 'gamma_Cz', 'gamma_C2', 'gamma_C4', 'gamma_CP4', 'gamma_P3',
       'gamma_Pz', 'gamma_P4', 'gamma_O1', 'gamma_Oz', 'gamma_O2'],
      dtype='object')
E:\Pruebas%20BCI\Completos10Hz\ElizabethMorales\S1
	|-pasivo
(3051,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Index(['beta_FC3', 'beta_FCz', 'beta_FC4', 'beta_

C:\Users\fercy\AppData\Local\Temp\ipykernel_6156\3322384139.py:13: RuntimeWarning: Mean of empty slice
  promedios = np.nanmean(promedios, axis=0)


(3149,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Index(['beta_FC3', 'beta_FCz', 'beta_FC4', 'beta_CP3', 'beta_C3', 'beta_C1',
       'beta_Cz', 'beta_C2', 'beta_C4', 'beta_CP4', 'beta_P3', 'beta_Pz',
       'beta_P4', 'beta_O1', 'beta_Oz', 'beta_O2'],
      dtype='object')
gamma Index(['gamma_FC3', 'gamma_FCz', 'gamma_FC4', 'gamma_CP3', 'gamma_C3',
       'gamma_C1', 'gamma_Cz', 'gamma_C2', 'gamma_C4', 'gamma_CP4', 'gamma_P3',
       'gamma_Pz', 'gamma_P4', 'gamma_O1', 'gamma_Oz', 'gamma_O2'],
      dtype='object')
E:\Pruebas%20BCI\Completos10Hz\SamualSanchez\S3
	|-pasivo
(3670,)
mu Index(['mu_FC3', 'mu_FCz', 'mu_FC4', 'mu_CP3', 'mu_C3', 'mu_C1', 'mu_Cz',
       'mu_C2', 'mu_C4', 'mu_CP4', 'mu_P3', 'mu_Pz', 'mu_P4', 'mu_O1', 'mu_Oz',
       'mu_O2'],
      dtype='object')
beta Index(['beta_FC3', 'beta_FCz', 'beta_FC4', 'beta_CP3

In [9]:
import mne

In [10]:

ch_types = ["eeg"] * 16
info = mne.create_info(list(ch_names), sfreq=10, ch_types=ch_types)
info.set_montage('standard_1020')

Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,19 points
Good channels,16 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,10.00 Hz
Highpass,0.00 Hz
Lowpass,5.00 Hz


In [11]:

def get_val2plot(actividad, reposo):
    # Promedio por canal/frecuencia sobre trials
    prom_act = np.nanmean(actividad, axis=0)
    prom_rest = np.nanmean(reposo, axis=0)

    # ERD/ERS en dB
    prom = 10 * (np.log10(prom_act) - np.log10(prom_rest))
    return prom

In [12]:
mult_mu_0 =  get_val2plot(mu_act0, mu_rep0)
mult_mu_5 = get_val2plot(mu_act5, mu_rep5)
mult_mu_10 = get_val2plot(mu_act10, mu_rep10)

mult_beta_0 =  get_val2plot(beta_act0, beta_rep0)
mult_beta_5 = get_val2plot(beta_act5, beta_rep5)
mult_beta_10 = get_val2plot(beta_act10, beta_rep10)

mult_gamma_0 =  get_val2plot(gamma_act0, gamma_rep0)
mult_gamma_5 = get_val2plot(gamma_act5, gamma_rep5)
mult_gamma_10 = get_val2plot(gamma_act10, gamma_rep10)


mini = np.min([mult_mu_0.min(), mult_beta_0.min(), mult_gamma_0.min(), mult_mu_5.min(), mult_beta_5.min(), mult_gamma_5.min(), mult_mu_10.min(), mult_beta_10.min(), mult_gamma_10.min()])
maxi = np.max([mult_mu_0.max(), mult_beta_0.max(), mult_gamma_0.max(), mult_mu_5.max(), mult_beta_5.max(), mult_gamma_5.max(), mult_mu_10.max(), mult_beta_10.max(), mult_gamma_10.max()])

In [13]:
%matplotlib inline

In [14]:

# You can further customize font to a sans-serif one
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
plt.rcParams['font.size'] = 12

In [29]:
%matplotlib qt

In [51]:

cmap_yop = "jet"

fig, axes = plt.subplots(3, 3, figsize=(6, 9))

data_list = [[mult_mu_10, mult_beta_10, mult_gamma_10],
             [mult_mu_5, mult_beta_5, mult_gamma_5],
             [mult_mu_0, mult_beta_0, mult_gamma_0]]

# Adjust the layout for better spacing
fig.subplots_adjust(left=0.08, right=0.9, top=0.9, bottom=0.1, hspace=0.2, wspace=0.2)

for i in range(3):
    for j in range(3):
        mne.viz.plot_topomap(data_list[i][j], info, axes=axes[i,j],
                             cmap=cmap_yop, vlim=(mini, maxi),
                             show=False, extrapolate='local', contours=5, size = 15)

# Color bar
cbar_ax = fig.add_axes([0.92, (0.6 / 2), 0.01, 0.4])
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap_yop, norm=plt.Normalize(vmin=mini,vmax=maxi)), cax=cbar_ax, label='ERD/ERS (dB)')

# Labels
fig.text(0.03, 0.78, 'SE10', va='center', ha='center', rotation='vertical', fontsize=20, fontweight='bold')
fig.text(0.04, 0.46, 'SE5', va='center', ha='center', rotation='vertical', fontsize=20, fontweight='bold')
fig.text(0.04, 0.20, 'AM', va='center', ha='center', rotation='vertical', fontsize=20, fontweight='bold')
fig.text(0.20, 0.96, r'$\mu$', va='center', ha='center', fontsize=30, fontweight='bold')
fig.text(0.49, 0.96, r'$\beta$', va='center', ha='center', fontsize=30, fontweight='bold')
fig.text(0.78, 0.96, r'$\gamma$', va='center', ha='center', fontsize=30, fontweight='bold')



# Líneas horizontales
#ax_div.axhline(y=0.95, color='gray', linestyle='-', linewidth=2)
#ax_div.axvline(x=0.08, color='gray', linestyle='--', linewidth=2)

plt.show()


In [16]:
data_dict = {
    'mu_0': mult_mu_0,
    'mu_5': mult_mu_5,
    'mu_10': mult_mu_10,
    'beta_0': mult_beta_0,
    'beta_5': mult_beta_5,
    'beta_10': mult_beta_10,
    'gamma_0': mult_gamma_0,
    'gamma_5': mult_gamma_5,
    'gamma_10': mult_gamma_10
}

df = pd.DataFrame(data_dict)
df.to_csv('ERD_ERS_results.csv', index=True)  